In [0]:
%sql
-- ══════════════════════════════════════════════════════════════════════════
-- AGENT 2: ALL TABLE CREATION SCRIPTS
-- Run in order, all in learning_dev.s819534
-- ══════════════════════════════════════════════════════════════════════════


-- ── 1. Layer Registry ──────────────────────────────────────────────────────

CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc2_layer_registry_b (

    layer_id                    STRING,

    source_fully_qualified_name STRING,

    layer_name                  STRING,
    layer_fully_qualified_name  STRING,
    layer_catalog               STRING,
    layer_schema                STRING,
    layer_table                 STRING,

    business_date_column        STRING,

    expected_survival_rate      DOUBLE,
    expected_latency_minutes    INT,

    reconciliation_metrics      STRING,

    is_active                   BOOLEAN,
    registered_timestamp        TIMESTAMP,
    registered_by               STRING

)
USING DELTA
COMMENT 'Registers each data layer for cross-layer reconciliation by Agent 2.';


-- ── 2. Reconciliation Run ──────────────────────────────────────────────────

CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc2_reconciliation_run_b (

    reconciliation_run_id           STRING,

    source_fully_qualified_name     STRING,
    business_date                   DATE,

    triggered_by                    STRING,
    agent1_validation_run_id        STRING,

    reconciliation_start_timestamp  TIMESTAMP,
    reconciliation_end_timestamp    TIMESTAMP,
    reconciliation_duration_seconds DOUBLE,

    overall_status                  STRING,
    total_checks_run                INT,
    total_checks_passed             INT,
    total_checks_warned             INT,
    total_checks_critical           INT,

    ai_enrichment_status            STRING,
    ai_enrichment_timestamp         TIMESTAMP,

    alert_sent                      BOOLEAN,
    alert_sent_timestamp            TIMESTAMP,
    alert_channel                   STRING,

    created_timestamp               TIMESTAMP,
    created_by                      STRING

)
USING DELTA
PARTITIONED BY (business_date)
COMMENT 'One record per reconciliation execution per source table per business date.';


-- ── 3. Reconciliation Results ──────────────────────────────────────────────

CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc2_reconciliation_results_b (

    result_id                   STRING,
    reconciliation_run_id       STRING,

    source_fully_qualified_name STRING,
    business_date               DATE,
    from_layer                  STRING,
    to_layer                    STRING,
    check_type                  STRING,

    metric_name                 STRING,
    check_status                STRING,

    source_value                DOUBLE,
    target_value                DOUBLE,
    expected_value              DOUBLE,
    deviation_percentage        DOUBLE,
    threshold_warning           DOUBLE,
    threshold_critical          DOUBLE,

    check_detail                STRING,

    agent1_overall_status       STRING,
    agent1_failure_category     STRING,

    ai_summary                  STRING,
    ai_recommended_action       STRING,
    ai_failure_category         STRING,
    ai_confidence_score         DOUBLE,
    ai_model_version            STRING,
    ai_invocation_id            STRING,

    created_timestamp           TIMESTAMP,
    ai_enriched_timestamp       TIMESTAMP

)
USING DELTA
PARTITIONED BY (business_date)
COMMENT 'One record per check per reconciliation run.';


-- ── 4. Silver Store Transactions ───────────────────────────────────────────

CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc2_silver_store_transactions (

    transaction_id              STRING,
    transaction_date            DATE,
    transaction_timestamp       TIMESTAMP,

    store_id                    STRING,
    store_region                STRING,
    terminal_id                 STRING,
    cashier_id                  STRING,

    customer_id                 STRING,
    loyalty_card_number         STRING,

    transaction_type            STRING,
    payment_method              STRING,
    channel                     STRING,

    gross_amount                DOUBLE,
    discount_amount             DOUBLE,
    tax_amount                  DOUBLE,
    net_amount                  DOUBLE,
    currency_code               STRING,

    item_count                  INT,
    quantity_total              INT,

    is_derived_net_amount       BOOLEAN,
    quality_score               DOUBLE,

    source_file_name            STRING,
    silver_load_timestamp       TIMESTAMP,
    pipeline_version            STRING

)
USING DELTA
PARTITIONED BY (transaction_date)
COMMENT 'Silver layer for store transactions. Deduplicated, VOID-excluded, nulls derived.';


-- ── 5. Silver Inventory Movements ─────────────────────────────────────────

CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc2_silver_inventory_movements (

    event_id                    STRING,
    event_timestamp             TIMESTAMP,
    event_type                  STRING,
    business_date               DATE,

    warehouse_id                STRING,
    store_id                    STRING,
    location_code               STRING,

    sku                         STRING,
    product_id                  STRING,
    category_code               STRING,
    supplier_id                 STRING,

    quantity_change             INT,
    quantity_before             INT,
    quantity_after              INT,
    unit_cost                   DOUBLE,
    total_cost_impact           DOUBLE,

    reference_document_id       STRING,
    reference_document_type     STRING,

    cdc_operation               STRING,
    cdc_source_system           STRING,
    cdc_sequence_number         BIGINT,

    is_deduped                  BOOLEAN,
    quality_flag                STRING,

    silver_load_timestamp       TIMESTAMP,
    pipeline_version            STRING

)
USING DELTA
PARTITIONED BY (business_date)
COMMENT 'Silver layer for inventory movements. DELETE-filtered, deduped, malformed-excluded.';


-- ── 6. Gold Daily Store Metrics ────────────────────────────────────────────

CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc2_gold_daily_store_metrics (

    store_id                    STRING,
    store_region                STRING,
    business_date               DATE,

    total_transactions          BIGINT,
    total_gross_amount          DOUBLE,
    total_discount_amount       DOUBLE,
    total_net_amount            DOUBLE,
    total_tax_amount            DOUBLE,
    avg_basket_value            DOUBLE,
    total_items_sold            BIGINT,
    unique_customers            BIGINT,
    return_count                BIGINT,
    return_rate                 DOUBLE,

    silver_load_timestamp       TIMESTAMP,
    gold_load_timestamp         TIMESTAMP,
    pipeline_version            STRING

)
USING DELTA
PARTITIONED BY (business_date)
COMMENT 'Gold layer: one row per store per business date aggregated from Silver transactions.';


-- ── 7. Gold Daily Inventory Summary ───────────────────────────────────────

CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc2_gold_daily_inventory_summary (

    warehouse_id                STRING,
    event_type                  STRING,
    business_date               DATE,

    event_count                 BIGINT,
    total_quantity_in           BIGINT,
    total_quantity_out          BIGINT,
    net_quantity_change         BIGINT,
    total_cost_impact           DOUBLE,
    unique_skus                 BIGINT,
    unique_stores               BIGINT,

    silver_load_timestamp       TIMESTAMP,
    gold_load_timestamp         TIMESTAMP,
    pipeline_version            STRING

)
USING DELTA
PARTITIONED BY (business_date)
COMMENT 'Gold layer: one row per warehouse per event type per date aggregated from Silver inventory.';


-- ── Verify all 7 tables created ────────────────────────────────────────────

SHOW TABLES IN learning_dev.s819534
LIKE 'uc2_*';

database,tableName,isTemporary
s819534,uc2_gold_daily_inventory_summary,false
s819534,uc2_gold_daily_store_metrics,false
s819534,uc2_layer_registry_b,false
s819534,uc2_reconciliation_results_b,false
s819534,uc2_reconciliation_run_b,false
s819534,uc2_silver_inventory_movements,false
s819534,uc2_silver_store_transactions,false


In [0]:
%sql
-- ── Register all layers for both pilot tables ──────────────────────────────

INSERT INTO learning_dev.s819534.uc2_layer_registry_b
VALUES

-- ── Store Transactions: RAW layer ─────────────────────────────────────────
(
    'lyr-txn-raw',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'RAW',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'learning_dev',
    's819534',
    'uc1_raw_store_transactions',
    'transaction_date',
    1.0,
    0,
    '["net_amount", "gross_amount", "discount_amount"]',
    true,
    current_timestamp(),
    'agent2_admin'
),

-- ── Store Transactions: SILVER layer ──────────────────────────────────────
(
    'lyr-txn-silver',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'SILVER',
    'learning_dev.s819534.uc2_silver_store_transactions',
    'learning_dev',
    's819534',
    'uc2_silver_store_transactions',
    'transaction_date',
    0.92,
    120,
    '["net_amount", "gross_amount", "discount_amount"]',
    true,
    current_timestamp(),
    'agent2_admin'
),

-- ── Store Transactions: GOLD layer ────────────────────────────────────────
(
    'lyr-txn-gold',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'GOLD',
    'learning_dev.s819534.uc2_gold_daily_store_metrics',
    'learning_dev',
    's819534',
    'uc2_gold_daily_store_metrics',
    'business_date',
    NULL,
    60,
    '["total_net_amount", "total_gross_amount", "total_discount_amount"]',
    true,
    current_timestamp(),
    'agent2_admin'
),

-- ── Inventory Movements: RAW layer ────────────────────────────────────────
(
    'lyr-inv-raw',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'RAW',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'learning_dev',
    's819534',
    'uc1_raw_inventory_movements',
    'business_date',
    1.0,
    0,
    '["quantity_change", "total_cost_impact"]',
    true,
    current_timestamp(),
    'agent2_admin'
),

-- ── Inventory Movements: SILVER layer ─────────────────────────────────────
(
    'lyr-inv-silver',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'SILVER',
    'learning_dev.s819534.uc2_silver_inventory_movements',
    'learning_dev',
    's819534',
    'uc2_silver_inventory_movements',
    'business_date',
    0.88,
    60,
    '["quantity_change", "total_cost_impact"]',
    true,
    current_timestamp(),
    'agent2_admin'
),

-- ── Inventory Movements: GOLD layer ───────────────────────────────────────
(
    'lyr-inv-gold',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'GOLD',
    'learning_dev.s819534.uc2_gold_daily_inventory_summary',
    'learning_dev',
    's819534',
    'uc2_gold_daily_inventory_summary',
    'business_date',
    NULL,
    30,
    '["net_quantity_change", "total_cost_impact"]',
    true,
    current_timestamp(),
    'agent2_admin'
);

-- Verify
SELECT
    source_fully_qualified_name,
    layer_name,
    layer_fully_qualified_name,
    expected_survival_rate,
    expected_latency_minutes,
    reconciliation_metrics
FROM learning_dev.s819534.uc2_layer_registry_b
ORDER BY source_fully_qualified_name, layer_name;

source_fully_qualified_name,layer_name,layer_fully_qualified_name,expected_survival_rate,expected_latency_minutes,reconciliation_metrics
learning_dev.s819534.uc1_raw_inventory_movements,GOLD,learning_dev.s819534.uc2_gold_daily_inventory_summary,null,30,"[""net_quantity_change"", ""total_cost_impact""]"
learning_dev.s819534.uc1_raw_inventory_movements,RAW,learning_dev.s819534.uc1_raw_inventory_movements,1.0,0,"[""quantity_change"", ""total_cost_impact""]"
learning_dev.s819534.uc1_raw_inventory_movements,SILVER,learning_dev.s819534.uc2_silver_inventory_movements,0.88,60,"[""quantity_change"", ""total_cost_impact""]"
learning_dev.s819534.uc1_raw_store_transactions,GOLD,learning_dev.s819534.uc2_gold_daily_store_metrics,null,60,"[""total_net_amount"", ""total_gross_amount"", ""total_discount_amount""]"
learning_dev.s819534.uc1_raw_store_transactions,RAW,learning_dev.s819534.uc1_raw_store_transactions,1.0,0,"[""net_amount"", ""gross_amount"", ""discount_amount""]"
learning_dev.s819534.uc1_raw_store_transactions,SILVER,learning_dev.s819534.uc2_silver_store_transactions,0.92,120,"[""net_amount"", ""gross_amount"", ""discount_amount""]"


In [0]:
%sql
CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc2_layer_check_config_b (

    layer_check_config_id       STRING,

    -- Scope: table + layer transition + check type
    source_fully_qualified_name STRING,
    from_layer                  STRING,    -- RAW, SILVER
    to_layer                    STRING,    -- SILVER, GOLD

    check_type                  STRING,    -- ROW_COUNT, AGGREGATE, FRESHNESS

    -- Default thresholds for this transition + check combination
    -- Individual metrics in uc2_reconciliation_metric_config_b
    -- inherit these unless they override with their own values
    default_warning_threshold   DOUBLE,
    default_critical_threshold  DOUBLE,
    threshold_unit              STRING,

    -- Description shown in UI
    description                 STRING,

    is_active                   BOOLEAN,
    created_timestamp           TIMESTAMP,
    created_by                  STRING,
    updated_timestamp           TIMESTAMP,
    updated_by                  STRING

)
USING DELTA
COMMENT 'Layer-aware default thresholds per check type per layer transition. Metrics inherit these unless individually overridden.';

In [0]:
%sql
INSERT INTO learning_dev.s819534.uc2_layer_check_config_b VALUES

-- ── uc1_raw_store_transactions ─────────────────────────────────────────────

-- RAW → SILVER
(
    'lcc-txn-rs-rowcount',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'RAW', 'SILVER', 'ROW_COUNT',
    5.0, 15.0, 'PERCENTAGE',
    'Warn if survival rate deviates more than 5% from expected 92%. Critical at 15%.',
    true, current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),
(
    'lcc-txn-rs-aggregate',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'RAW', 'SILVER', 'AGGREGATE',
    2.0, 10.0, 'PERCENTAGE',
    'Warn if financial aggregate drops more than 2% from Raw to Silver. VOID and null store_id exclusions may cause small drops.',
    true, current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),
(
    'lcc-txn-rs-freshness',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'RAW', 'SILVER', 'FRESHNESS',
    90.0, 120.0, 'MINUTES',
    'Batch table expected within 2 hours. Warn at 90 min, critical at 120 min.',
    true, current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),

-- SILVER → GOLD
(
    'lcc-txn-sg-rowcount',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'SILVER', 'GOLD', 'ROW_COUNT',
    0.0, 5.0, 'PERCENTAGE',
    'Gold must have exactly one row per store. Any missing store row is critical.',
    true, current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),
(
    'lcc-txn-sg-aggregate',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'SILVER', 'GOLD', 'AGGREGATE',
    0.01, 0.5, 'PERCENTAGE',
    'Silver to Gold must match exactly. Any deviation indicates Gold calculation error.',
    true, current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),
(
    'lcc-txn-sg-freshness',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'SILVER', 'GOLD', 'FRESHNESS',
    90.0, 120.0, 'MINUTES',
    'Gold should be ready within 1 hour of Silver.',
    true, current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),

-- ── uc1_raw_inventory_movements ───────────────────────────────────────────

-- RAW → SILVER
(
    'lcc-inv-rs-rowcount',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'RAW', 'SILVER', 'ROW_COUNT',
    5.0, 15.0, 'PERCENTAGE',
    'Warn if survival rate deviates more than 5% from expected 88%. DELETE exclusions cause natural drop.',
    true, current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),
(
    'lcc-inv-rs-aggregate',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'RAW', 'SILVER', 'AGGREGATE',
    15.0, 25.0, 'PERCENTAGE',
    'Wider threshold for inventory. DELETE CDC exclusions cause natural aggregate reduction of 5-15%.',
    true, current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),
(
    'lcc-inv-rs-freshness',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'RAW', 'SILVER', 'FRESHNESS',
    45.0, 60.0, 'MINUTES',
    'Near-real-time stream. Silver should be ready within 15 minutes. Warn at 45, critical at 60.',
    true, current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),

-- SILVER → GOLD
(
    'lcc-inv-sg-rowcount',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'SILVER', 'GOLD', 'ROW_COUNT',
    0.0, 5.0, 'PERCENTAGE',
    'Gold must have exactly one row per warehouse per event type. Any missing combo is critical.',
    true, current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),
(
    'lcc-inv-sg-aggregate',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'SILVER', 'GOLD', 'AGGREGATE',
    0.01, 0.5, 'PERCENTAGE',
    'Silver to Gold must match exactly. Any deviation indicates Gold aggregation error.',
    true, current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),
(
    'lcc-inv-sg-freshness',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'SILVER', 'GOLD', 'FRESHNESS',
    45.0, 60.0, 'MINUTES',
    'Gold should be ready within 30 minutes of Silver for inventory.',
    true, current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
);

-- Verify
SELECT
    SUBSTRING_INDEX(source_fully_qualified_name, '.', -1) AS table_name,
    from_layer,
    to_layer,
    check_type,
    default_warning_threshold   AS warn,
    default_critical_threshold  AS crit,
    threshold_unit              AS unit
FROM learning_dev.s819534.uc2_layer_check_config_b
ORDER BY
    source_fully_qualified_name,
    from_layer, to_layer, check_type;

table_name,from_layer,to_layer,check_type,warn,crit,unit
uc1_raw_inventory_movements,RAW,SILVER,AGGREGATE,15.0,25.0,PERCENTAGE
uc1_raw_inventory_movements,RAW,SILVER,FRESHNESS,45.0,60.0,MINUTES
uc1_raw_inventory_movements,RAW,SILVER,ROW_COUNT,5.0,15.0,PERCENTAGE
uc1_raw_inventory_movements,SILVER,GOLD,AGGREGATE,0.01,0.5,PERCENTAGE
uc1_raw_inventory_movements,SILVER,GOLD,FRESHNESS,45.0,60.0,MINUTES
uc1_raw_inventory_movements,SILVER,GOLD,ROW_COUNT,0.0,5.0,PERCENTAGE
uc1_raw_store_transactions,RAW,SILVER,AGGREGATE,2.0,10.0,PERCENTAGE
uc1_raw_store_transactions,RAW,SILVER,FRESHNESS,90.0,120.0,MINUTES
uc1_raw_store_transactions,RAW,SILVER,ROW_COUNT,5.0,15.0,PERCENTAGE
uc1_raw_store_transactions,SILVER,GOLD,AGGREGATE,0.01,0.5,PERCENTAGE


In [0]:
%sql


CREATE TABLE IF NOT EXISTS learning_dev.s819534.uc2_reconciliation_metric_config_b (

    metric_config_id            STRING,

    -- Scope: which table, which layer transition, which check
    source_fully_qualified_name STRING,
    from_layer                  STRING,    -- RAW, SILVER
    to_layer                    STRING,    -- SILVER, GOLD
    check_type                  STRING,    -- ROW_COUNT, AGGREGATE, FRESHNESS

    -- The specific metric being configured
    metric_name                 STRING,
    -- For ROW_COUNT: "row_count_survival_rate"
    -- For AGGREGATE: "net_amount", "gross_amount", "discount_amount"
    -- For AGGREGATE Silver→Gold: "net_amount_vs_total_net_amount"
    -- For FRESHNESS: "silver_data_presence", "gold_data_presence"

    -- Source and target column mapping (for AGGREGATE only)
    source_column               STRING,    -- column in FROM layer
    target_column               STRING,    -- column in TO layer (can differ)

    -- Thresholds for this specific metric
    warning_threshold           DOUBLE,
    critical_threshold          DOUBLE,
    threshold_unit              STRING,    -- PERCENTAGE, COUNT, MINUTES

    -- Whether this metric check is active
    is_active                   BOOLEAN,

    -- Display order in results
    display_order               INT,

    -- Audit
    created_timestamp           TIMESTAMP,
    created_by                  STRING,
    updated_timestamp           TIMESTAMP,
    updated_by                  STRING

)
USING DELTA
COMMENT 'Configures specific metrics and thresholds for each Agent 2 reconciliation check. One row per metric per layer transition per source table.';

In [0]:
%sql
INSERT INTO learning_dev.s819534.uc2_reconciliation_metric_config_b VALUES

-- ══════════════════════════════════════════════════════════════════════════
-- raw_store_transactions
-- ══════════════════════════════════════════════════════════════════════════

-- ── RAW → SILVER Row Count ────────────────────────────────────────────────
(
    'mc-txn-rs-rowcount',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'RAW', 'SILVER', 'ROW_COUNT',
    'row_count_survival_rate',
    NULL, NULL,
    5.0, 15.0, 'PERCENTAGE',
    true, 1,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),

-- ── RAW → SILVER Aggregates ───────────────────────────────────────────────
(
    'mc-txn-rs-net-amount',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'RAW', 'SILVER', 'AGGREGATE',
    'net_amount',
    'net_amount', 'net_amount',
    2.0, 10.0, 'PERCENTAGE',
    true, 2,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),
(
    'mc-txn-rs-gross-amount',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'RAW', 'SILVER', 'AGGREGATE',
    'gross_amount',
    'gross_amount', 'gross_amount',
    2.0, 10.0, 'PERCENTAGE',
    true, 3,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),
(
    'mc-txn-rs-discount-amount',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'RAW', 'SILVER', 'AGGREGATE',
    'discount_amount',
    'discount_amount', 'discount_amount',
    2.0, 10.0, 'PERCENTAGE',
    true, 4,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),

-- ── RAW → SILVER Freshness ────────────────────────────────────────────────
(
    'mc-txn-rs-freshness',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'RAW', 'SILVER', 'FRESHNESS',
    'silver_data_presence',
    NULL, NULL,
    90.0, 120.0, 'MINUTES',
    true, 5,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),

-- ── SILVER → GOLD Row Count ───────────────────────────────────────────────
(
    'mc-txn-sg-rowcount',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'SILVER', 'GOLD', 'ROW_COUNT',
    'gold_row_count',
    NULL, NULL,
    0.0, 5.0, 'PERCENTAGE',
    true, 6,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),

-- ── SILVER → GOLD Aggregates ──────────────────────────────────────────────
(
    'mc-txn-sg-net-amount',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'SILVER', 'GOLD', 'AGGREGATE',
    'net_amount_vs_total_net_amount',
    'net_amount', 'total_net_amount',
    0.01, 0.5, 'PERCENTAGE',
    true, 7,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),
(
    'mc-txn-sg-gross-amount',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'SILVER', 'GOLD', 'AGGREGATE',
    'gross_amount_vs_total_gross_amount',
    'gross_amount', 'total_gross_amount',
    0.01, 0.5, 'PERCENTAGE',
    true, 8,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),
(
    'mc-txn-sg-discount-amount',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'SILVER', 'GOLD', 'AGGREGATE',
    'discount_amount_vs_total_discount_amount',
    'discount_amount', 'total_discount_amount',
    0.01, 0.5, 'PERCENTAGE',
    true, 9,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),

-- ── SILVER → GOLD Freshness ───────────────────────────────────────────────
(
    'mc-txn-sg-freshness',
    'learning_dev.s819534.uc1_raw_store_transactions',
    'SILVER', 'GOLD', 'FRESHNESS',
    'gold_data_presence',
    NULL, NULL,
    90.0, 120.0, 'MINUTES',
    true, 10,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),

-- ══════════════════════════════════════════════════════════════════════════
-- raw_inventory_movements
-- ══════════════════════════════════════════════════════════════════════════

-- ── RAW → SILVER Row Count ────────────────────────────────────────────────
(
    'mc-inv-rs-rowcount',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'RAW', 'SILVER', 'ROW_COUNT',
    'row_count_survival_rate',
    NULL, NULL,
    5.0, 15.0, 'PERCENTAGE',
    true, 1,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),

-- ── RAW → SILVER Aggregates ───────────────────────────────────────────────
(
    'mc-inv-rs-qty-change',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'RAW', 'SILVER', 'AGGREGATE',
    'quantity_change',
    'quantity_change', 'quantity_change',
    15.0, 25.0, 'PERCENTAGE',
    true, 2,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),
(
    'mc-inv-rs-cost-impact',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'RAW', 'SILVER', 'AGGREGATE',
    'total_cost_impact',
    'total_cost_impact', 'total_cost_impact',
    15.0, 25.0, 'PERCENTAGE',
    true, 3,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),

-- ── RAW → SILVER Freshness ────────────────────────────────────────────────
(
    'mc-inv-rs-freshness',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'RAW', 'SILVER', 'FRESHNESS',
    'silver_data_presence',
    NULL, NULL,
    45.0, 60.0, 'MINUTES',
    true, 4,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),

-- ── SILVER → GOLD Row Count ───────────────────────────────────────────────
(
    'mc-inv-sg-rowcount',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'SILVER', 'GOLD', 'ROW_COUNT',
    'gold_row_count',
    NULL, NULL,
    0.0, 5.0, 'PERCENTAGE',
    true, 5,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),

-- ── SILVER → GOLD Aggregates ──────────────────────────────────────────────
(
    'mc-inv-sg-qty-change',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'SILVER', 'GOLD', 'AGGREGATE',
    'quantity_change_vs_net_quantity_change',
    'quantity_change', 'net_quantity_change',
    0.01, 0.5, 'PERCENTAGE',
    true, 6,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),
(
    'mc-inv-sg-cost-impact',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'SILVER', 'GOLD', 'AGGREGATE',
    'total_cost_impact_vs_total_cost_impact',
    'total_cost_impact', 'total_cost_impact',
    0.01, 0.5, 'PERCENTAGE',
    true, 7,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
),

-- ── SILVER → GOLD Freshness ───────────────────────────────────────────────
(
    'mc-inv-sg-freshness',
    'learning_dev.s819534.uc1_raw_inventory_movements',
    'SILVER', 'GOLD', 'FRESHNESS',
    'gold_data_presence',
    NULL, NULL,
    45.0, 60.0, 'MINUTES',
    true, 8,
    current_timestamp(), 'agent2_admin',
    current_timestamp(), 'agent2_admin'
);

-- Verify
SELECT
    SUBSTRING_INDEX(source_fully_qualified_name, '.', -1) AS source_table,
    from_layer,
    to_layer,
    check_type,
    metric_name,
    source_column,
    target_column,
    warning_threshold,
    critical_threshold,
    threshold_unit,
    is_active
FROM learning_dev.s819534.uc2_reconciliation_metric_config_b
ORDER BY
    source_fully_qualified_name,
    from_layer,
    to_layer,
    check_type,
    display_order;

source_table,from_layer,to_layer,check_type,metric_name,source_column,target_column,warning_threshold,critical_threshold,threshold_unit,is_active
uc1_raw_inventory_movements,RAW,SILVER,AGGREGATE,quantity_change,quantity_change,quantity_change,15.0,25.0,PERCENTAGE,true
uc1_raw_inventory_movements,RAW,SILVER,AGGREGATE,total_cost_impact,total_cost_impact,total_cost_impact,15.0,25.0,PERCENTAGE,true
uc1_raw_inventory_movements,RAW,SILVER,FRESHNESS,silver_data_presence,null,null,45.0,60.0,MINUTES,true
uc1_raw_inventory_movements,RAW,SILVER,ROW_COUNT,row_count_survival_rate,null,null,5.0,15.0,PERCENTAGE,true
uc1_raw_inventory_movements,SILVER,GOLD,AGGREGATE,quantity_change_vs_net_quantity_change,quantity_change,net_quantity_change,0.01,0.5,PERCENTAGE,true
uc1_raw_inventory_movements,SILVER,GOLD,AGGREGATE,total_cost_impact_vs_total_cost_impact,total_cost_impact,total_cost_impact,0.01,0.5,PERCENTAGE,true
uc1_raw_inventory_movements,SILVER,GOLD,FRESHNESS,gold_data_presence,null,null,45.0,60.0,MINUTES,true
uc1_raw_inventory_movements,SILVER,GOLD,ROW_COUNT,gold_row_count,null,null,0.0,5.0,PERCENTAGE,true
uc1_raw_store_transactions,RAW,SILVER,AGGREGATE,net_amount,net_amount,net_amount,2.0,10.0,PERCENTAGE,true
uc1_raw_store_transactions,RAW,SILVER,AGGREGATE,gross_amount,gross_amount,gross_amount,2.0,10.0,PERCENTAGE,true
